<a href="https://colab.research.google.com/github/navap3206-debug/hello-world/blob/main/Bootstrap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Carlos Santiago Nava Pulido

In [138]:
from google.colab import files
import pandas as pd
from sklearn.utils import resample
import numpy as np

import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score


# CLASE 1

## EDA

In [69]:
dt = files.upload()

Saving Motor Trend Car Road Tests.xlsx to Motor Trend Car Road Tests (1).xlsx


In [70]:
df = pd.read_excel('Motor Trend Car Road Tests.xlsx')

In [71]:
df.head(1)

,model,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,Mazda RX4,21.0,6,160.0,110,3.9,2.62,16.46,0,1,4,4


In [72]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 12 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   model   32 non-null     object 
 1   mpg     32 non-null     float64
 2   cyl     32 non-null     int64  
 3   disp    32 non-null     float64
 4   hp      32 non-null     int64  
 5   drat    32 non-null     float64
 6   wt      32 non-null     float64
 7   qsec    32 non-null     float64
 8   vs      32 non-null     int64  
 9   am      32 non-null     int64  
 10  gear    32 non-null     int64  
 11  carb    32 non-null     int64  
dtypes: float64(5), int64(6), object(1)
memory usage: 3.1+ KB


## **REALIZAR** REGRESIÓN LINEAL PREDICIENDO "MPG" Y USANDO HP Y QSEC


In [50]:
Y = ['mpg']
numerical_col = ["hp","qsec"]

In [51]:
X = df[numerical_col]
Y = df[Y]

INTERVALOS DE CONFIANZA


In [52]:
import statsmodels.api as sm
X_sm = sm.add_constant(X)

In [53]:
sm_model = sm.OLS(Y, X_sm)
sm_results = sm_model.fit()

print(sm_results.summary())

                            OLS Regression Results                            
Dep. Variable:                    mpg   R-squared:                       0.637
Model:                            OLS   Adj. R-squared:                  0.612
Method:                 Least Squares   F-statistic:                     25.43
Date:                Thu, 30 Apr 2026   Prob (F-statistic):           4.18e-07
Time:                        22:56:35   Log-Likelihood:                -86.170
No. Observations:                  32   AIC:                             178.3
Df Residuals:                      29   BIC:                             182.7
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         48.3237     11.103      4.352      0.0

## SACAR EL STANDARD ERROR

In [29]:

e = sm_results.resid
e.head(1)

,0
0,-3.42537


In [30]:
sigma = (e.T @ e) / (len(e) - X_sm.shape[1])
sigma

np.float64(14.099785332780655)

In [31]:
var = sigma * np.linalg.inv(X_sm.T.dot(X_sm))
var

array([[ 1.23283411e+02, -1.22628430e-01, -5.87462625e+00],
       [-1.22628430e-01,  1.94123319e-04,  5.27504533e-03],
       [-5.87462625e+00,  5.27504533e-03,  2.85781527e-01]])

In [34]:
SE = np.sqrt(np.diag(var))
SE

array([11.10330633,  0.01393281,  0.53458538])

## INTERVALOS POR VARIOS METODOS:

In [35]:
interval_b0 = [sm_results.params[0] - 2 * SE[0], sm_results.params[0] + 2 * SE[0]]
interval_b1 = [sm_results.params[1] - 2 * SE[1], sm_results.params[1] + 2 * SE[1]]
interval_b2 = [sm_results.params[2] - 2 * SE[2], sm_results.params[2] + 2 * SE[2]]

In [37]:
print(interval_b0)
print(interval_b1)
print(interval_b2)

[np.float64(26.117092516519424), np.float64(70.53031782174969)]
[np.float64(-0.11245867264236552), np.float64(-0.05672741470582039)]
[np.float64(-1.9557503826808793), np.float64(0.1825911334123258)]


In [32]:
#intervalos de confianza proceso
se_b0 = sm_results.bse[0]
se_b1 = sm_results.bse[1]
se_b2 = sm_results.bse[2]

interval_b0 = [sm_results.params[0] - 2 * se_b0, sm_results.params[0] + 2 * se_b0]
interval_b1 = [sm_results.params[1] - 2 * se_b1, sm_results.params[1] + 2 * se_b1]
interval_b2 = [sm_results.params[2] - 2 * se_b2, sm_results.params[2] + 2 * se_b2]

print(interval_b0)
print(interval_b1)
print(interval_b2)



[np.float64(26.117092516520273), np.float64(70.53031782174884)]
[np.float64(-0.11245867264236498), np.float64(-0.05672741470582093)]
[np.float64(-1.955750382680837), np.float64(0.1825911334122834)]


In [33]:
#CONFIRMAMOS CON LA LIBRERIA
conf_int = sm_results.conf_int(alpha=0.05)
display(conf_int)

,0,1
const,25.614894,71.032516
hp,-0.113089,-0.056097
qsec,-1.979929,0.206770


## BOOTSTRAP

In [38]:
n_iterations = 1000
b0 = []
b1 = []
b2 = []

for i in range(n_iterations):
    boot = resample(df, replace=True, n_samples=len(df))
    X = boot[numerical_col]
    Y = boot[Y.columns]
    X_sm = sm.add_constant(X)
    sm_model = sm.OLS(Y, X_sm)
    sm_results = sm_model.fit()
    b0.append(sm_results.params[0])
    b1.append(sm_results.params[1])
    b2.append(sm_results.params[2])




MEDIAS  Y DESVIACIONES DE CADA BETA

In [39]:
mean_b0 = np.mean(b0)
std_b0 = np.std(b0)
mean_b1 = np.mean(b1)
std_b1 = np.std(b1)
mean_b2 = np.mean(b2)
std_b2 = np.std(b2)



In [40]:
#intervalos
print(mean_b0 - 2 * std_b0, mean_b0 + 2 * std_b0)
print(mean_b1 - 2 * std_b1, mean_b1 + 2 * std_b1)
print(mean_b2 - 2 * std_b2, mean_b2 + 2 * std_b2)

28.015061884198953 71.64292527078273
-0.12035341472969413 -0.054335689758073145
-2.000263294188885 0.0957947528545422


COMPARACIÓN

# CLASE 2 (ENSAMBLE DE MODELOS)

In [110]:
df.head(1)

,model,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,Mazda RX4,21.0,6,160.0,110,3.9,2.62,16.46,0,1,4,4


In [111]:
X = ['cyl','disp','hp','drat','wt','qsec','vs','am','gear','carb']
y = ["mpg"]


train test

In [156]:
X_train, X_test, Y_train, Y_test = train_test_split(df[X],df[y], test_size=.5,random_state=42)

BUCLE DE MODELOS

In [157]:
vars = []
models = []
for i in range(1000):
  columnas_selected = np.random.choice(X, size=3, replace=False)
  X_selected_with_const = sm.add_constant(X_train[columnas_selected])
  sm_model = sm.OLS(Y_train, X_selected_with_const)
  sm_results = sm_model.fit()
  vars.append(sm_results)
  models.append(sm_results)

In [158]:
vars = np.array(vars)

BUCLE DE PREDICCIONES

In [159]:
predicciones = []
for i in range(1000):
  columnas = models[i].model.exog_names[1:]
  X_test_for_prediction = sm.add_constant(X_test[columnas])
  prediccion = models[i].predict(X_test_for_prediction)
  predicciones.append(prediccion)


In [160]:
predicciones = np.array(predicciones)
prom_pred = predicciones.mean(axis=0)

In [161]:
predicciones

array([[24.21516354, 10.39001262, 12.25237025, ..., 16.10745053,
        16.10745053, 13.80433493],
       [24.21516354, 10.39001262, 12.25237025, ..., 16.10745053,
        16.10745053, 13.80433493],
       [20.08432682, 14.98430602, 14.00821844, ..., 15.21248234,
        14.45189461, 11.92927866],
       ...,
       [23.6724378 , 10.72301357, 12.29672826, ..., 17.12565876,
        16.75504488, 12.99149687],
       [23.24597117,  9.70971044, 12.61458423, ..., 17.74227367,
        17.74227367, 13.99327068],
       [19.89499667, 13.54786868, 13.74665191, ..., 13.721804  ,
        13.721804  , 15.36176566]])

In [151]:
#promedio por fila
prom_pred

array([20.28890106, 10.13867677, 14.43728049, 27.13527911, 23.58075922,
       20.1106769 , 13.54528868, 27.49077782, 15.30040246, 21.74234405,
       15.47587954, 10.37137519, 19.79116665, 15.26420348, 14.74101002,
       13.49543305])

In [154]:
r2 = r2_score(Y_test, prom_pred)

In [155]:
r2

0.7833571432555588